# 0. Setup: environment, drives, sample data: solutions

Exercises of [chapter 0](../00_setup.md). Run the setup cell first: it makes the sample data root.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: find the repository root, make the sample data root if needed, use the sample's site registry
import os, subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))
SAMPLE = ROOT / "output/manual/sample_root"
if not SAMPLE.exists():
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"
EXTRACT = ROOT / "docs/manual/data/montana_extract"
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")
import warnings; warnings.simplefilter("ignore")
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import subprocess
from data_acquisition.catalog import summary as summaries

**1.** List the files of the sample data root with their sizes. Which file is the largest, and why?

In [ ]:
for p in sorted(SAMPLE.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>9,d}  {p.relative_to(SAMPLE)}")
# The orthomosaic is the largest: four bands of noisy colour texture, which compress poorly. The DSM is tiny
# although it holds 4-byte heights: a smooth surface compresses to almost nothing. The GeoPackage's ~100 KB is
# mostly the SQLite container (pages, indexes, metadata tables), not its two dams.

**2.** Print the versions of GDAL and PROJ inside rasterio, of PROJ inside pyproj, and of the system's `ogr2ogr`. How many different versions of PROJ do you have?

In [ ]:
import pyproj, rasterio
print("rasterio: GDAL", rasterio.__gdal_version__, "| PROJ", rasterio.__proj_version__)
print("pyproj:   PROJ", pyproj.proj_version_str)
print("system:  ", subprocess.run(["ogr2ogr", "--version"], capture_output=True, text=True).stdout.strip())
# Usually three copies of GDAL/PROJ: rasterio's, pyproj's and the system's. Chapter 2 shows why it matters.

**3.** Read `pytest.ini`: which plugins are disabled, and why?

In [ ]:
print((ROOT / "pytest.ini").read_text())
# The ROS plugins (launch_testing, launch_ros, ament_*): ROS adds them to the Python path of every terminal on
# machines where it is installed, and they are incompatible with this version of pytest.

**4.** Print the path of the site registry the pipeline uses, with and without `BEAVER_SITES`, and the sites in each.

In [ ]:
print("with BEAVER_SITES:   ", os.environ["BEAVER_SITES"], list(summaries.load_sites()))
saved = os.environ.pop("BEAVER_SITES")
print("without BEAVER_SITES:", summaries.SITES, len(summaries.load_sites()), "sites")
os.environ["BEAVER_SITES"] = saved